# Φάση Δ: Advanced Technique

**Επιλογή 1 από:**
- Association Rules (Apriori / FP-Growth)
- Clustering (K-Means)
- Άλλο

Εμείς επιλέξαμε το K-Means

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import PCA

print("Αρχικοποίηση SparkSession...")
spark = SparkSession.builder \
    .appName("Stroke_Advanced") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Φόρτωση Gold Layer...")
train_gold = spark.read.parquet("../data/train_gold.parquet")
train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))

print("Εκπαίδευση K-Means Clustering...")
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)
kmeans_model = kmeans.fit(train_gold)
kmeans_preds = kmeans_model.transform(train_gold)

print("Εφαρμογή PCA για οπτικοποίηση των Clusters...")
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(kmeans_preds)
kmeans_pca = pca_model.transform(kmeans_preds)

print("Αποθήκευση προβλέψεων στο δίσκο...")
# ΠΡΟΣΟΧΗ: Κρατάμε και τη στήλη 'features' για τον ορθό υπολογισμό του Silhouette Score
kmeans_pca.select("stroke", "cluster", "pca_features", "features") \
    .write.mode("overwrite").parquet("../data/preds_kmeans.parquet")

print("Το K-Means εκπαιδεύτηκε και αποθηκεύτηκε επιτυχώς!")